# 02 — Model 1: B0 TF-IDF + LinearSVC

**Part of the 5-notebook split** of `code-mixed-hate-speech-detection-pipeline-v2.ipynb`.
The complete original notebook is kept unchanged for upload.

**Scope:** Baseline B0 only (3.1–3.3): char 2–5 gram TF-IDF + LinearSVC across 3 seeds.

**Prerequisites:** 01_data_and_preprocessing.ipynb (processed splits).

**Writes:** b0_seed*_*.npy, split_state/02_b0_state.json

> Run 01 → 05 in order in the same environment: Kaggle with **Internet ON** and
> **GPU T4 x2**. Every assigned code cell is copied verbatim from the original;
> only the small handoff cells are new. Outputs are intentionally cleared — the
> executed results live in the unchanged original notebook, the PDF, and the
> `results/` artifacts.

### 0.1 Dependencies

In [ ]:
import sys, subprocess

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# transformers>=4.46 required: eval_strategy + processing_class were introduced in 4.46.
pip_install([
    "transformers>=4.46,<5",
    "datasets>=3.0",
    "accelerate>=0.34",
    "scipy>=1.11",
])

### 0.2 Imports & Global Configuration (single source of truth for the environment)

In [ ]:
import os, re, json, math, time, random, hashlib, shutil, warnings, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset as TorchDataset

from datasets import load_dataset
from huggingface_hub import HfApi
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, set_seed as hf_set_seed,
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    f1_score, precision_recall_fscore_support, matthews_corrcoef,
    balanced_accuracy_score, confusion_matrix, classification_report,
)
from scipy import stats

warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.dpi"] = 110

# ---- Environment contract: the ONLY place run mode is decided ---------------
SMOKE_TEST = False   # True = tiny data + tiny model, CPU fast pass (~2 min)

PRISM_ID  = "pankajbiswas6/prism-hinglish-hate-speech"
MEHATE_REPO = "l3cube-pune/MarathiNLP"          # GitHub repo, CSV fetched at HEAD SHA
MEHATE_CSV  = "MeEval/L3Cube-MeHate/codemix-hate-small-test.csv"

SEEDS = [42, 43, 44]
TINY_MODEL = "sshleifer/tiny-distilroberta-base"
MODELS = {
    "hing-roberta": "l3cube-pune/hing-roberta",
    "muril":        "google/muril-base-cased",
    "xlm-roberta":  "xlm-roberta-base",
}
if SMOKE_TEST:
    MODELS = {k: TINY_MODEL for k in MODELS}

MAX_LEN     = 128 if not SMOKE_TEST else 32
NUM_EPOCHS  = 3   if not SMOKE_TEST else 1
BATCH_SIZE  = 32  if not SMOKE_TEST else 4
BOOTSTRAP_N = 1_000 if not SMOKE_TEST else 100
SMOKE_N     = 200
LR, WEIGHT_DECAY, WARMUP_RATIO = 2e-5, 0.01, 0.1   # named, not magic

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./kaggle_working")
RESULTS_DIR, FIGURES_DIR, PREDS_DIR = WORK_DIR/"results", WORK_DIR/"figures", WORK_DIR/"preds"
for d in (RESULTS_DIR, FIGURES_DIR, PREDS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Fail fast: the full run is impossible on CPU and would silently drop fp16.
if not SMOKE_TEST:
    assert torch.cuda.is_available(), (
        "GPU required: set Accelerator to 'GPU T4 x2' (or P100) in notebook settings. "
        "For a CPU pass, set SMOKE_TEST = True in section 0.2.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"SMOKE_TEST = {SMOKE_TEST} | Device = {DEVICE} | Work dir = {WORK_DIR.resolve()}")

### 0.3 Reproducibility helpers

In [ ]:
def set_seed(seed: int) -> None:
    """Seed python, numpy and torch (incl. CUDA). Note: with fp16 + cuDNN this is
    reproducible-in-practice, not bit-exact; `full_determinism=True` would cost speed."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

ALL_SEED_RESULTS = []   # one row per (model, seed, split)
PER_CLASS_RESULTS = []  # per-class P/R/F1 rows
COSTS = []              # params / train time / ms-per-item per (model, seed)
THRESH_CURVES = {}      # (model_key, seed) -> (thresholds, val macro-F1 curve)
LOG_HISTORY = {}        # (model_key, seed) -> trainer.state.log_history
TRAINED = {}            # (model_key, seed=SEEDS[0]) -> artifacts needed in sections 6-7

print("Reproducibility helpers ready.")

In [ ]:
# ---- Shared handoff directory for the 5-part notebook chain ------------------
# Run 01 -> 02 -> 03 -> 04 -> 05 in the SAME environment (Kaggle, Internet ON, GPU).
STATE_DIR = WORK_DIR / "split_state"
STATE_DIR.mkdir(parents=True, exist_ok=True)
print("Split state dir:", STATE_DIR)

### Shared data (verbatim handoff)

In [ ]:
# ---- Handoff from 01_data_and_preprocessing.ipynb ---------------------------
# Reconstruct the exact processed `prism` splits the monolith held in memory.
prism = {}
for _k in ("train", "val", "test"):
    _p = RESULTS_DIR / f"prism_{_k}.csv"
    assert _p.exists(), f"Missing {_p} - run 01_data_and_preprocessing.ipynb first"
    _df = pd.read_csv(_p)
    _df["text"] = _df["text"].fillna("").astype(str)
    _df["label"] = _df["label"].astype(int)
    prism[_k] = _df
print("Loaded processed splits:", {k: len(v) for k, v in prism.items()})

## 3. Baseline B0 — Char n-gram TF-IDF + Linear SVM

### 3.1 Unified metric + model definition

One `compute_metrics` serves B0 and every transformer (v1 defined it twice).

In [ ]:
def compute_metrics(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return {"macro_f1": f, "macro_precision": p, "macro_recall": r,
            "mcc": matthews_corrcoef(y_true, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_true, y_pred)}

def per_class_rows(y_true, y_pred, model_label, key, seed, split):
    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    rows = []
    for cls, name in (("0", "non-hate"), ("1", "hate")):
        m = rep[cls]
        rows.append({"model": model_label, "key": key, "seed": seed, "split": split,
                     "class": name, "precision": m["precision"], "recall": m["recall"],
                     "f1": m["f1-score"], "support": m["support"]})
    return rows

def record_run(model_label, key, seed, split, y_true, y_pred, extras=None):
    metrics = compute_metrics(y_true, y_pred)
    ALL_SEED_RESULTS.append({"model": model_label, "key": key, "seed": seed,
                             "split": split, **metrics, **(extras or {})})
    PER_CLASS_RESULTS.extend(per_class_rows(y_true, y_pred, model_label, key, seed, split))
    return metrics

def make_b0(seed: int) -> Pipeline:
    # Char 2-5 gram TF-IDF + LinearSVC. GPU-free, robust to romanisation noise.
    return Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(2, 5), lowercase=True,
                                  min_df=2, max_features=300_000, sublinear_tf=True)),
        ("clf", LinearSVC(C=1.0, random_state=seed)),
    ])

X_train, y_train = prism["train"]["text"].to_numpy(), prism["train"]["label"].to_numpy()
X_val,   y_val   = prism["val"]["text"].to_numpy(),   prism["val"]["label"].to_numpy()
X_test,  y_test  = prism["test"]["text"].to_numpy(),  prism["test"]["label"].to_numpy()
print(f"train {len(X_train)} | val {len(X_val)} | test {len(X_test)}")

### 3.2 Train and evaluate across 3 seeds (checkpointed per seed)

In [ ]:
for seed in SEEDS:
    set_seed(seed)
    model = make_b0(seed)
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    train_s = time.perf_counter() - t0
    t1 = time.perf_counter()
    val_pred, test_pred = model.predict(X_val), model.predict(X_test)
    infer_ms = (time.perf_counter() - t1) / len(X_test) * 1000

    val_m  = record_run("B0 TF-IDF+SVM", "b0", seed, "val",  y_val,  val_pred)
    test_m = record_run("B0 TF-IDF+SVM", "b0", seed, "test", y_test, test_pred)
    COSTS.append({"model": "B0 TF-IDF+SVM", "key": "b0", "seed": seed,
                  "params_m": np.nan, "train_seconds": train_s, "ms_per_item": infer_ms})
    np.save(PREDS_DIR / f"b0_seed{seed}_val.npy", val_pred)
    np.save(PREDS_DIR / f"b0_seed{seed}_test.npy", test_pred)
    print(f"[b0 seed {seed}]  val macro-F1 {val_m['macro_f1']:.4f} | "
          f"test macro-F1 {test_m['macro_f1']:.4f} | train {train_s:.1f}s")

### 3.3 Baseline summary

In [ ]:
b0_test = [r for r in ALL_SEED_RESULTS if r["key"] == "b0" and r["split"] == "test"]
print(f"B0 test macro-F1 across seeds: {np.mean([r['macro_f1'] for r in b0_test]):.4f} "
      f"± {np.std([r['macro_f1'] for r in b0_test]):.4f}")

In [ ]:
# ---- Split handoff (02_b0 -> 05) ----------------------------------------------
def _jsonable(x):
    # numpy scalars -> plain python so json.dumps keeps numbers numeric
    if isinstance(x, dict):
        return {str(k): _jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_jsonable(v) for v in x]
    if isinstance(x, np.generic):
        return x.item()
    return x

_curves = {}
_logs = {}
(STATE_DIR / "02_b0_state.json").write_text(json.dumps(_jsonable({
    "all_seed": ALL_SEED_RESULTS,
    "per_class": PER_CLASS_RESULTS,
    "costs": COSTS,
    "thresh_curves": _curves,
    "log_history": _logs,
}), indent=2))
print("Saved 02_b0_state.json:", len(ALL_SEED_RESULTS), "result rows,", len(COSTS), "cost rows")